![image.png](https://i.imgur.com/a3uAqnb.png)
# Lab 2: Speech-to-Text (STT / ASR)

This notebook explores **automatic speech recognition** — from the classical Bayes formulation and WER metric to acoustic features and Whisper inference.

You will implement evaluation code, visualize MFCCs, and run a foundation ASR model on real audio.

> 💡 ASR quality is measured at the **word** level (WER), but models operate on **frames** of acoustic features. Bridge that gap in your answers.

__Let's set up the environment.__ Install dependencies below, then run the import cell.



# 📦 Installing Required Python Libraries

This cell installs packages needed for this lab.

- **PyTorch / Torchaudio** — Audio datasets (e.g. LibriSpeech) and tensor ops.
- **Transformers** — Whisper and ASR pipelines.
- **Librosa** — MFCC and log-mel feature extraction.
- **Matplotlib / NumPy** — Spectrogram visualization.


# 📥 Importing Essential Python Libraries

Standard imports for Part B programming exercises and GPU checks.


---

## 💻 Part A — Introduction & Preliminaries
__Let's implement the core ideas in PyTorch.__



### 🛠️ A1. Word Error Rate (WER) implementation

### 🛠️ A2. Acoustic features

Extract **MFCC** or **log-mel** features and visualize an utterance.

#### 👀 Commentary (B2)

**MFCCs** emphasize **short-term spectral envelope** shaped by the vocal tract, capturing **phonetic/articulatory** information (formant-like cues). They de-emphasize fine pitch harmonics (via mel + DCT compression), making them robust for **phone discrimination**.

### 🛠️ A3. Whisper transcription

Run OpenAI **Whisper** (via `transformers`) on a short clip.


# Part B: Non-English ASR

In this part of this assignment, you will leverage  large, pretrained speech models of choice to do ASR non-English languages. For this part, are free to use any model available on HuggingFace.


We will be looking at these four languages in this section:


*   Lingala (`ln_cd`)
*   Korean(`ko_kr`)
*   isiXhosa (`xh_za`)
*   isiZulu (`zu_za`)
*   Irish (`ga_ie`)



##### Part B Imports

## Inference example

Below is an example of how to get the WER of a loaded dataset:

This code runs inference with a single model for the test set of a single chosen language. You can use this as a starting point to run inference and evaluations on different models and languages.

You can change the model you evaluate by changing the `model_name` variable.

As an example, we will continue to use the Telugu test set from FLEURS.

## B.1 Filter dataset by utterance length

### B.1.1 Add input length to dataset

 `add_input_length` to adds the length of the utterance to the dataset with the attribute `input_length`.


### B.1.2 Task: Remove utterances longer than 30 seconds

Let's filter out utterances longer than 30 seconds out of the dataset.Write a function called `is_audio_length_in_range` to filters out utterances over 30 seconds using the `input_length` that is now in your dataset.

### B.1.3 Task: Make length-based subsets of the dataset (3 points)

Let's make subsets of the dataset based on utterance length. For these datasets we have three utterance lengths:

*   Short (0-10 seconds)
*   Medium (10 - 16 seconds)
*   Long (16 - 30 seconds)


Write a funntion to turn a dataset into a `small`, `medium`, and `longer` dataset based on the utterance lengths above.

### B.1.4 Task: Make length-based datasets for our 5 languages (5 points)

Now do this for each of the 5 languages we will be working with:


*   Lingala (`ln_cd`)
*   Korean(`ko_kr`)
*   isiXhosa (`xh_za`)
*   isiZulu (`zu_za`)
*   Irish (`ga_ie`)


Return a `dataset_length_dict` with keys being the lowercase language names and values being `[num_short, num_med, num_long]` utterances per language.

## B.2 Task: Compute aggregate word error rate across several languages and utterance lengths



Compute the word error rate of the test set for each of these languages across different utterance lengths.

Utterance lengths:


*   Short (0-10 seconds)
*   Medium (10 - 16 seconds)
*   Long (16 - 30 seconds)

Languages:

*   Lingala (`ln_cd`)
*   Korean(`ko_kr`)
*   isiXhosa (`xh_za`)
*   isiZulu (`zu_za`)
*   Irish (`ga_ie`)


Start with MMS 1B `facebook/mms-1b-all` and see if you can find a better model that reduces the WER on medium length utterances across all languages.


We would like a table of WER by utterance length per language for at least two publicly available models.

### Step-by-step guide to 2.2
For those who find it helpful, here is a **step-by-step guide** to creating a basic code block that runs:

**1 Environment preparation**
Start each session by making sure you have a GPU runtime and modern libraries.

* Upgrade the core stack via !pip install -U "transformers>=4.40" sentencepiece soundfile

* Restart the runtime so the new version of transformers is picked up.

* Authenticate once per runtime if you plan on touching Common Voice or gated MMS files:
  from huggingface_hub import login → paste your write token.

**2 Pull the FLEURS test split and attach duration**
For each target language code (e.g. ln_cd for Lingala)

1. Load the "test" split.

2. Immediately cast the audio column to a 16 kHz sampling rate – MMS assumes that rate.

3. Compute a dur field in seconds (len(array) / sampling_rate) via .map.

That dataset object will be reused for every model and length bucket.

*Gotcha*: forgetting the Audio(sampling_rate=16_000) cast will silently resample at runtime and make you wait forever.

**3 Duration buckets**
Use duration buckets you have created above.

**4 ASR pipelines**
**4 ASR pipeline (facebook/mms-1b-all)**
To create a working pipeline:

Load facebook/mms-1b-all with target_lang=<ISO-639-3> via AutoProcessor + AutoModelForCTC; then build a pipeline with the returned tokenizer, feature extractor and model.

Debug checklist

* If every WER is ~1.0 you probably forgot to pass target_lang when loading facebook/mms-1b-all.

**5 Batch inference and WER**
Inside the double-for-loop language × duration-bin:

1. Call the pipeline on bucket["audio"] with a modest batch size (8 fits into a 12 GB V100).

2. Lower-case both hypotheses and references before computing WER via evaluate.load("wer").

3. Append a record {language, len_bin, model, WER} to a running Python list.

Print progress as you go so you can see which combination is slow or failing.

**6 Pivot and inspect**
After the loops finish, convert the list into a pandas.DataFrame and call


df.pivot_table(values="WER",
               index=["language", "len_bin"]).sort_index()

That single line produces exactly the table the graders expect: one row per language-bin, one column per model, lower = better.

**7 Typical outcomes & interpretation**
* MMS-1B-all plus the correct adapter yields WER ≈ 0.20 – 0.40 on these languages.

* The assignment asks you to beat MMS on the medium bucket → obvious next step is to swap in language-specific MMS checkpoints or a community fine-tune.

Use those observations, not my numbers, when you write your reflection.

**8 Trouble-shooting cheatsheet**

* Endless download loops → nuke the faulty cache folder under ~/.cache/huggingface/hub and re-run.

* Runtime slows to a crawl → confirm the “Device set to use cuda:0” banner; Colab occasionally drops back to CPU.

* Tokenizer error on MMS after a restart → recreate the temp vocab; the directory is ephemeral.